# 24. AdaBoost
**Use case:** Oil & Gas — Drilling stuck-pipe early warning

Stuck pipe is one of the costliest drilling problems: freeing the string or sidetracking can cost hundreds of thousands of dollars and days of rig time. Drillers watch hookload, torque, rate of penetration, mud weight and standpipe pressure, but early signs are subtle and easy to miss on a long shift. The operations centre already streams these parameters every few seconds from each rig. Success means an alarm that catches most precursors early enough to circulate or ream, with a false-alarm rate the crew will tolerate, validated on wells the model has never seen.

> 💡 Think of a team of junior bank tellers screening cheques: the first checks one obvious thing, the next pays extra attention to the cheques the first one misjudged, and so on. At the end they vote, with the more reliable tellers getting a louder voice.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## EASY: Three stumps that spot stuck pipe
Run 3 rounds of AdaBoost by hand on 10 drilling snapshots and watch the row weights shift toward hard cases.

| 5W + H | |
|---|---|
| **Who** | A drilling engineer learning how boosting combines simple rules into an alarm. |
| **What** | Three one-question rules (decision stumps) on torque, hookload and ROP, weighted and combined into one vote. |
| **When** | At each round the algorithm re-weights rows it got wrong before training the next stump. |
| **Where** | Ten hand-typed rig snapshots in a numpy array, no ML library. |
| **Why** | Seeing the weight updates explains why AdaBoost focuses on the tricky cases and how alpha sets each rule's say. |
| **How** | Find the stump with the lowest weighted error, compute alpha, boost weights of misclassified rows, and sum alpha-weighted votes. |

**🧠 ML concepts:** Decision stump (weak learner), Weighted error, Learner weight alpha, Sample re-weighting, Weighted majority vote  
**🐍 Python concepts:** 2-D numpy arrays, np.full and np.unique, np.where as vectorised if/else, Nested for loops, Functions returning tuples, Conditional expressions inside f-strings

In [ ]:
# AdaBoost by hand: 3 rounds of decision stumps on 10 drilling snapshots (1 = stuck pipe followed)
import numpy as np

# 🐍 PY: each row = [torque_kftlb, hookload_klb, rop_ft_hr]; a 2-D numpy array is like a small table
X = np.array([
    [12, 210, 85], [13, 215, 80], [18, 260, 40], [11, 205, 90], [19, 250, 35],
    [17, 240, 70], [14, 262, 72], [12, 255, 75], [15, 265, 30], [16, 220, 42],
])
# 🧠 ML: AdaBoost uses labels -1 / +1 (here +1 = stuck pipe within the next hour)
y = np.array([-1, -1, 1, -1, 1, -1, 1, -1, 1, -1])
names = ["torque", "hookload", "rop"]
n = len(y)
# 🧠 ML: start with equal weight on every row - all mistakes count the same
w = np.full(n, 1 / n)


def best_stump(X, y, w):
    """Try every feature/threshold/direction; return the stump with the lowest WEIGHTED error."""
    best = None
    for f in range(X.shape[1]):
        # 🐍 PY: np.unique gives sorted distinct values to use as candidate thresholds
        for thr in np.unique(X[:, f]):
            for sign in (1, -1):
                # 🐍 PY: np.where(cond, a, b) is a vectorised if/else over the whole column
                pred = np.where(sign * (X[:, f] - thr) >= 0, 1, -1)
                err = w[pred != y].sum()
                if best is None or err < best[0]:
                    best = (err, f, thr, sign, pred)
    return best


ensemble_score = np.zeros(n)
for rnd in range(1, 4):
    err, f, thr, sign, pred = best_stump(X, y, w)
    # 🧠 ML: alpha = how much say this stump gets; lower error -> bigger vote
    alpha = 0.5 * np.log((1 - err) / max(err, 1e-10))
    rule = f"{names[f]} {'>=' if sign == 1 else '<='} {thr}"
    print(f"Round {rnd}: stump '{rule}'  weighted_error={err:.3f}  alpha={alpha:.3f}")
    # 🧠 ML: boost weights of rows this stump got WRONG so the next stump focuses on them
    w = w * np.exp(-alpha * y * pred)
    # 🐍 PY: dividing an array by its sum normalises it so weights add up to 1
    w = w / w.sum()
    print("   new weights:", " ".join(f"{v:.2f}" for v in w))
    ensemble_score += alpha * pred

# 🧠 ML: final prediction = sign of the weighted vote of all stumps
final = np.sign(ensemble_score)
print("\nrow | torque hookload rop | actual | vote   | predicted")
for i in range(n):
    print(f" {i:2d} | {X[i, 0]:5d} {X[i, 1]:8d} {X[i, 2]:4d} |   {y[i]:+d}   | {ensemble_score[i]:+.2f}  |   {int(final[i]):+d}")
print(f"\nTraining accuracy: {(final == y).mean():.0%}")

**Try this next:**
- Run 1 round only: which rows are wrong?
- Change row 6's ROP to 40 and see how the chosen stumps change.
- Add a fourth feature column (mud weight) and update names accordingly.

## MEDIUM: AdaBoostClassifier on 4,000 rig snapshots
Train scikit-learn's AdaBoostClassifier on synthetic drilling data, tune the alarm threshold, and read feature importance.

| 5W + H | |
|---|---|
| **Who** | A real-time drilling operations centre wanting an early warning for stuck pipe. |
| **What** | 200 boosted stumps scoring each 1-minute snapshot of hookload, torque, ROP, mud weight, standpipe pressure and torque trend. |
| **When** | Every minute while drilling, so the driller has time to circulate or ream before the pipe sticks. |
| **Where** | Data from the rig's surface sensors via WITSML into the operations centre, simulated here with numpy/pandas. |
| **Why** | A stuck-pipe event can cost hundreds of thousands of dollars in fishing and sidetrack time; a warning lets the crew act early. |
| **How** | Stratified split, AdaBoost with depth-1 trees, ROC-AUC, comparison of two thresholds, classification report and feature importances. |

**🧠 ML concepts:** AdaBoost with decision stumps, Rare-event classification, Stratified splitting, ROC-AUC, Threshold tuning for recall, Feature importance  
**🐍 Python concepts:** Dict of arrays to DataFrame, Vectorised pandas arithmetic, astype for type conversion, Unpacking ravel() into four variables, Series.sort_values and .items(), Keyword arguments

In [ ]:
# AdaBoostClassifier: early warning of stuck pipe from 1-minute drilling parameters
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)
N = 4000

# ---- Synthetic rig-sensor snapshots (one row = a 1-minute summary while drilling) ----
# 🐍 PY: a dict of numpy arrays turns straight into a DataFrame, one key per column
df = pd.DataFrame({
    "hookload_klb": rng.normal(230, 15, N),
    "torque_kftlb": rng.normal(14, 2.5, N),
    "rop_ft_hr": rng.normal(70, 18, N).clip(5, None),
    "mud_weight_ppg": rng.normal(10.5, 0.6, N),
    "standpipe_psi": rng.normal(3200, 250, N),
    "torque_trend_10min": rng.normal(0, 0.8, N),      # change in torque over last 10 minutes
})
# 🧠 ML: hidden "true" risk - rising torque, high hookload (overpull), falling ROP and heavy mud raise risk
logit = (-4.5 + 2.0 * df.torque_trend_10min + 0.10 * (df.hookload_klb - 230)
         - 0.05 * (df.rop_ft_hr - 70) + 1.5 * (df.mud_weight_ppg - 10.5)
         + 0.003 * (df.standpipe_psi - 3200))
# 🐍 PY: 1 / (1 + exp(-x)) is the logistic function; rng.random() < p draws a 0/1 outcome
df["stuck_pipe_next_hr"] = (rng.random(N) < 1 / (1 + np.exp(-logit))).astype(int)
print(f"Rows: {N}   stuck-pipe rate: {df.stuck_pipe_next_hr.mean():.1%}")

X = df.drop(columns="stuck_pipe_next_hr")
y = df.stuck_pipe_next_hr
# 🧠 ML: stratify=y keeps the same rare-event rate in train and test
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

# 🧠 ML: AdaBoost = many depth-1 trees ("stumps"), each focusing on rows the previous ones got wrong
model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=200,
    learning_rate=0.5,
    random_state=42,
)
model.fit(X_tr, y_tr)

# 🧠 ML: predict_proba gives a risk score; we use it for AUC and a custom alarm threshold
proba = model.predict_proba(X_te)[:, 1]
print(f"Test ROC-AUC: {roc_auc_score(y_te, proba):.3f}")

# 🧠 ML: AdaBoost probabilities bunch up near 0.5, so small threshold moves matter a lot
# 🧠 ML: with rare events the default 0.5 cut-off misses many; lower it to raise recall
for thr in (0.5, 0.45):
    pred = (proba >= thr).astype(int)
    # 🐍 PY: .ravel() flattens the 2x2 confusion matrix so we can unpack 4 numbers at once
    tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
    print(f"\nThreshold {thr}: caught {tp}/{tp + fn} events, {fp} false alarms")
    if thr == 0.45:
        print(classification_report(y_te, pred, target_names=["normal", "stuck"], digits=3))

# 🧠 ML: AdaBoost's feature_importances_ = how much weighted error each feature's stumps removed
imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature importance:")
# 🐍 PY: .items() yields (index, value) pairs from a pandas Series
for name, v in imp.items():
    print(f"  {name:20s} {v:.3f} " + "#" * int(v * 60))

print("\nBusiness read-out: lowering the threshold from 0.5 to 0.45 roughly doubles events caught;"
      "\neach false alarm costs a short pause to circulate/ream, far cheaper than a fishing job.")

**Try this next:**
- Set learning_rate=1.0 and n_estimators=50: how does AUC change?
- Try threshold 0.40 and count the extra false alarms.
- Drop torque_trend_10min from X and see how much AUC falls.

## HARD: Well-grouped tuning, GBM comparison, cost-based alarms
Tune AdaBoost with well-grouped cross-validation, inspect its staged learning curve, compare with GradientBoosting, and set the alarm threshold by dollar cost.

| 5W + H | |
|---|---|
| **Who** | A drilling analytics team taking a stuck-pipe model from prototype to a rig advisory tool. |
| **What** | A tuned AdaBoost model, a weighted variant for imbalance, a GradientBoosting challenger, a cost-optimal threshold and permutation importance. |
| **When** | Retrained per field campaign; scores every minute while drilling. |
| **Where** | 24 wells of synthetic data; 18 for training and CV, 6 held out entirely as unseen wells. |
| **Why** | A model that only works on wells it has seen, or picks a threshold by accuracy, will fail on the next rig or flood the driller with alarms. |
| **How** | GroupKFold by well, PR-AUC scoring, staged_predict_proba learning curve, balanced sample weights, cost curve over thresholds and joblib save. |

**🧠 ML concepts:** Grouped cross-validation to avoid leakage, PR-AUC for rare events, n_estimators vs learning_rate trade-off, Staged predictions as a learning curve, Class imbalance with sample weights, Cost-sensitive threshold selection  
**🐍 Python concepts:** Module-level constants, pd.concat of a list of frames, Dict comprehension filtering get_params(), np.quantile for threshold grids, Sorting with key=lambda, joblib.dump of a dict bundle

In [ ]:
# Production-style stuck-pipe early warning: AdaBoost tuned with grouped CV (by well), imbalance weights,
# staged learning curve, GradientBoosting comparison, cost-based alarm threshold, permutation importance.
import os
import tempfile
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")
COST_MISSED_EVENT = 250_000   # $ - fishing job + sidetrack rig time when stuck pipe is not prevented
COST_FALSE_ALARM = 4_000      # $ - ~1 hour of rig time spent circulating/reaming unnecessarily
FEATURES = ["hookload_klb", "torque_kftlb", "rop_ft_hr", "mud_weight_ppg", "standpipe_psi",
            "torque_trend_10min", "hookload_trend_10min"]


def make_wells(n_wells=24, rows_per_well=250, seed=42):
    rng = np.random.default_rng(seed)
    frames = []
    for w in range(n_wells):
        # 🧠 ML: each well has its own baseline (depth, formation) - rows from one well are correlated
        base_hl, base_mw = rng.normal(230, 20), rng.normal(10.5, 0.5)
        n = rows_per_well
        d = pd.DataFrame({
            "well_id": w,
            "hookload_klb": rng.normal(base_hl, 12, n),
            "torque_kftlb": rng.normal(14, 2.5, n),
            "rop_ft_hr": rng.normal(70, 18, n).clip(5, None),
            "mud_weight_ppg": rng.normal(base_mw, 0.3, n),
            "standpipe_psi": rng.normal(3200, 250, n),
            "torque_trend_10min": rng.normal(0, 0.8, n),
            "hookload_trend_10min": rng.normal(0, 4, n),
        })
        # 🧠 ML: risk depends on CHANGES vs the well's own normal, plus an interaction (overpull AND slow ROP)
        logit = (-5.0 + 1.8 * d.torque_trend_10min + 0.25 * d.hookload_trend_10min
                 + 1.2 * (d.mud_weight_ppg - 10.5) + 1.5 * ((d.rop_ft_hr < 45) & (d.hookload_trend_10min > 3)))
        d["stuck_pipe_next_hr"] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
        frames.append(d)
    # 🐍 PY: pd.concat stacks a list of DataFrames vertically into one table
    return pd.concat(frames, ignore_index=True)


def expected_cost(y_true, proba, thr):
    pred = proba >= thr
    missed = int(((~pred) & (y_true == 1)).sum())
    false_alarms = int((pred & (y_true == 0)).sum())
    return missed * COST_MISSED_EVENT + false_alarms * COST_FALSE_ALARM, missed, false_alarms


if __name__ == "__main__":
    df = make_wells()
    # 🧠 ML: hold out whole WELLS for testing - a random row split would leak each well's baseline
    test_wells = df.well_id >= 18
    train, test = df[~test_wells], df[test_wells]
    X_tr, y_tr, g_tr = train[FEATURES], train.stuck_pipe_next_hr, train.well_id
    X_te, y_te = test[FEATURES], test.stuck_pipe_next_hr
    print(f"Train: {len(train)} rows / 18 wells, event rate {y_tr.mean():.1%}  |  "
          f"Test: {len(test)} rows / 6 wells, event rate {y_te.mean():.1%}")

    ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), random_state=42)
    grid = {"n_estimators": [50, 150], "learning_rate": [0.1, 0.5],
            "estimator__max_depth": [1, 2]}
    # 🧠 ML: GroupKFold keeps every row of a well in the same fold -> honest CV for unseen wells
    # 🧠 ML: average precision (PR-AUC) is the right metric when positives are rare
    search = GridSearchCV(ada, grid, cv=GroupKFold(n_splits=3), scoring="average_precision", n_jobs=1)
    search.fit(X_tr, y_tr, groups=g_tr)
    print("\n== AdaBoost grid search (3-fold GroupKFold, PR-AUC) ==")
    res = pd.DataFrame(search.cv_results_)
    cols = ["param_n_estimators", "param_learning_rate", "param_estimator__max_depth", "mean_test_score"]
    print(res[cols].sort_values("mean_test_score", ascending=False).head(5).round(3).to_string(index=False))
    best_ada = search.best_estimator_

    # 🧠 ML: imbalance option - 'balanced' sample weights up-weight rare stuck-pipe rows during training
    sw = compute_sample_weight("balanced", y_tr)
    ada_bal = AdaBoostClassifier(**{k: v for k, v in best_ada.get_params().items()
                                    if k in ("estimator", "n_estimators", "learning_rate", "random_state")})
    ada_bal.fit(X_tr, y_tr, sample_weight=sw)

    # 🧠 ML: staged_predict_proba replays the ensemble one stump at a time = free learning curve
    print("\n== Staged learning curve on test wells (ROC-AUC as stumps are added) ==")
    n_total = best_ada.n_estimators
    checkpoints = sorted({1, 5, 20, 50, n_total // 2, n_total})
    for i, p in enumerate(best_ada.staged_predict_proba(X_te), start=1):
        if i in checkpoints:
            auc = roc_auc_score(y_te, p[:, 1])
            print(f"  {i:4d} estimators  AUC={auc:.3f} " + "#" * int((auc - 0.5) * 80))

    # 🧠 ML: gradient boosting fits residuals with a smooth loss - often more robust to label noise
    gb = GradientBoostingClassifier(n_estimators=150, max_depth=2, learning_rate=0.05,
                                    subsample=0.8, random_state=42)
    gb.fit(X_tr, y_tr, sample_weight=sw)

    print("\n== Model comparison on 6 unseen wells ==")
    models = {"AdaBoost (tuned)": best_ada, "AdaBoost + balanced": ada_bal, "GradientBoosting": gb}
    probas = {}
    for name, m in models.items():
        probas[name] = m.predict_proba(X_te)[:, 1]
        print(f"  {name:20s} ROC-AUC={roc_auc_score(y_te, probas[name]):.3f}  "
              f"PR-AUC={average_precision_score(y_te, probas[name]):.3f}")

    # 🧠 ML: ranking metrics barely move with weights - weighting mainly shifts the score scale (threshold)
    # 🧠 ML: pick the alarm threshold by $ cost, not accuracy - a missed event costs ~60x a false alarm
    print("\n== Alarm threshold by expected cost (AdaBoost) ==")
    p = probas["AdaBoost (tuned)"]
    # 🐍 PY: np.quantile picks thresholds spread across the score range, avoiding empty bins
    thresholds = np.unique(np.round(np.quantile(p, np.linspace(0.3, 0.98, 8)), 3))
    rows = [(t, *expected_cost(y_te.to_numpy(), p, t)) for t in thresholds]
    for t, c, m, fa in rows:
        print(f"  thr={t:.3f}  missed={m:3d}  false_alarms={fa:4d}  cost=${c / 1e6:5.2f}M")
    best_thr, best_cost, _, _ = min(rows, key=lambda r: r[1])
    no_model = int(y_te.sum()) * COST_MISSED_EVENT
    print(f"  -> threshold {best_thr:.3f}: ${best_cost / 1e6:.2f}M vs ${no_model / 1e6:.2f}M with no warning system")

    # 🧠 ML: permutation importance = drop in PR-AUC when a feature is shuffled (model-agnostic)
    pi = permutation_importance(best_ada, X_te, y_te, scoring="average_precision", n_repeats=3,
                                random_state=42, n_jobs=1)
    print("\n== Permutation importance (PR-AUC drop) ==")
    for f, v in sorted(zip(FEATURES, pi.importances_mean), key=lambda x: -x[1]):
        print(f"  {f:22s} {v:+.3f}")

    print("\nSafety note: this is an ADVISORY tool - the driller keeps authority (API RP 59 well-control"
          "\npractice, BSEE SEMS). Log every alarm and response; re-validate per field and formation.")
    path = os.path.join(tempfile.mkdtemp(), "stuck_pipe_ada.joblib")
    # 🐍 PY: joblib.dump serialises the whole fitted sklearn object, plus metadata in a dict
    joblib.dump({"model": best_ada, "threshold": float(best_thr), "features": FEATURES}, path)
    print(f"Saved model + threshold -> {os.path.basename(path)} in a temp dir")

**Try this next:**
- Replace GroupKFold with a plain KFold and compare the CV score with the test score.
- Change COST_MISSED_EVENT to 50,000 and see the optimal threshold move.
- Add learning_rate 1.0 back into the grid and look at the staged curve for overfitting.